In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))    # chạy notebook từ thư mục notebooks/

from src.config import MODELS, REPORTS, set_seed
from src.data import nap_du_lieu
from src.report import (TEN_HIEN_THI, bang_hitl, bang_ket_qua, cap_nham, doc_ket_qua,
                        ve_attention, ve_confusion, ve_so_sanh)
from src.train import build_transformer, nap_vocab, tao_vectorizer
set_seed()

In [ ]:
df = bang_ket_qua(TEN_HIEN_THI)
print(df.to_string(index=False))
ve_so_sanh(df)

In [ ]:
ve_confusion(TEN_HIEN_THI)
cap = {ten: cap_nham(ten) for ten in TEN_HIEN_THI}
for ten, c in cap.items():
    print(TEN_HIEN_THI[ten]); print(c.to_string(index=False), "\n")

In [ ]:
data = nap_du_lieu()
vec = tao_vectorizer(vocab=nap_vocab())
model = build_transformer()
model.load_weights(MODELS / "transformer_tu_xay.weights.h5")

# mỗi lớp lấy 1 tin đầu tiên trong test: Business, Sci/Tech, World
mau = [next(t for t, y in zip(data["texts"]["test"], data["y"]["test"]) if y == c) for c in (2, 3, 0)]
ve_attention(model, vec, mau)

In [ ]:
hitl = {}
for ten in ("transformer_tu_xay", "distilbert"):
    _, y_true, _, proba = doc_ket_qua(ten)
    hitl[ten] = bang_hitl(y_true, proba)
    print(TEN_HIEN_THI[ten]); print(hitl[ten].to_string(index=False), "\n")

In [ ]:
md = ["## So sanh 4 cach", df.to_markdown(index=False)]
for ten, c in cap.items():
    md += [f"### Cap nham nhieu nhat: {TEN_HIEN_THI[ten]}", c.to_markdown(index=False)]
for ten, h in hitl.items():
    md += [f"### Human-in-the-loop: {TEN_HIEN_THI[ten]}", h.to_markdown(index=False)]
(REPORTS / "ket_qua.md").write_text("\n\n".join(md), encoding="utf-8")
print(sorted(p.name for p in REPORTS.iterdir()))